### SafeTensor File Map

In [1]:
from safetensors import safe_open
import json

file_location = "model.safetensors"
with safe_open(file_location, framework="pt", device="cpu") as f:

    model_structure = {layer_name: f.get_tensor(layer_name).shape for layer_name in f.keys()}

    with open("modelSafetensors.json", "w") as file:
        json.dump(model_structure, file, indent=3)

    for name, shape in model_structure.items():
        print(f"Name: {name} | Shape: {shape}")

Name: model.embed_tokens.weight | Shape: torch.Size([157696, 896])
Name: model.layers.0.input_layernorm.weight | Shape: torch.Size([896])
Name: model.layers.0.mlp.down_proj.weight | Shape: torch.Size([896, 4864])
Name: model.layers.0.mlp.gate_proj.weight | Shape: torch.Size([4864, 896])
Name: model.layers.0.mlp.up_proj.weight | Shape: torch.Size([4864, 896])
Name: model.layers.0.post_attention_layernorm.weight | Shape: torch.Size([896])
Name: model.layers.0.self_attn.k_proj.bias | Shape: torch.Size([128])
Name: model.layers.0.self_attn.k_proj.weight | Shape: torch.Size([128, 896])
Name: model.layers.0.self_attn.o_proj.weight | Shape: torch.Size([896, 896])
Name: model.layers.0.self_attn.q_proj.bias | Shape: torch.Size([896])
Name: model.layers.0.self_attn.q_proj.weight | Shape: torch.Size([896, 896])
Name: model.layers.0.self_attn.v_proj.bias | Shape: torch.Size([128])
Name: model.layers.0.self_attn.v_proj.weight | Shape: torch.Size([128, 896])
Name: model.layers.1.input_layernorm.weig

### Attention

In [ ]:
import torch


class OutettsAttention(torch.nn.Module):
    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["hidden_size"]
        self.num_attn_heads = config["num_attention_heads"]
        self.num_kv_heads = config["num_key_value_heads"]
        self.attn_dim = 64
        self.max_seq_len = 512

        self.q_proj = torch.nn.Linear(
            in_features=self.d_model, out_features=self.d_model
        )
        self.k_proj = torch.nn.Linear(
            in_features=self.d_model,
            out_features=self.attn_dim * self.num_kv_heads,
        )
        self.v_proj = torch.nn.Linear(
            in_features=self.d_model,
            out_features=self.attn_dim * self.num_kv_heads,
        )
        self.o_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(self.max_seq_len, self.max_seq_len),
                diagonal=1,
            ).bool(),
        )

    def forward(self, x):

        B, S, _ = x.shape

        q = (
            self.q_proj(x)
            .view(B, S, self.num_attn_heads, self.attn_dim)
            .transpose(1, 2)
        )
        k = self.k_proj(x).view(B, S, self.num_kv_heads, self.attn_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_kv_heads, self.attn_dim).transpose(1, 2)

        num_groups = self.num_attn_heads // self.num_kv_heads

        k = k.repeat_interleave(num_groups, dim=1)
        v = v.repeat_interleave(num_groups, dim=1)

        scores = q @ k.transpose(-2, -1) * (self.attn_dim**-0.5)

        if S > 1:
            scores = scores.masked_fill(self.mask[:S, :S], float("-inf"))

        wei = torch.softmax(scores, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(B, S, self.d_model)

        return out

In [33]:
with open("config.json", "r") as f:
    config_file = json.load(f)

files = [
    safe_open(file, framework="pt", device="cpu") for file in ["model.safetensors"]
]

dummy_input = torch.rand(4, 12, config_file['hidden_size'])    

attn_block = OutettsAttention(config_file, 0, files)
output = attn_block(dummy_input)
output.shape

torch.Size([4, 12, 896])